# AI Project
This will be split into parts (below).  Comments will describe the code used and why it has been used:
* Part 1 - Installation of the new enviroment and libraries
* Part 2 - Creating the folder structures and implementing the data scrape
* Part 3 - Utilising the API key
* Part 4 - Creating the Q+A document
* Part 5 - Chunking the web scraped data, create a vector store for the Q+A and check distance scores
* Part 6 - Load the model, upload the context to the chatbot and enable the formatting of the response
* Part 7 - Ensuring the FAQ database is saved on my hard drive
* Part 8 - Prompt to check that questions can be answered as expected
* Part 9 - Install Streamlit

You will notice that some library imports repeat themselves. This is because I want you to see which libraries were involved with eadch part.  For effciency, I could I kept all the import libraires near the top of the notebook but this can make troubleshooting a bit more difficult.

## Part 1 - Installation of the new enviroment and libraries

In [ ]:
# Create the virtual environment. This allows me to use a new kernal with a localised directory stucture. It can help to prevent dependancy drift and global namespace pollution
!python -m venv rag_env

# Upgrade pip inside the new environment. I want to ensure that my packages within rag_env are as up to date as they can be preventing installation of package issues

!rag_env\Scripts\pip install --upgrade pip

# Install all required RAG, API Key Protection and AI packages - I tried using Chroma but this conflicted with my oneDrive so I switched to faiss
!rag_env\Scripts\pip install requests beautifulsoup4 faiss-cpu langchain langchain-community langchain-google-genai pandas streamlit python-dotenv scikit-learn

# Install text splitter. Digesting all the text all at once is too much so this library will chunk up blocks of text
!rag_env\Scripts\pip install langchain-text-splitters



In [ ]:
# This ensures that the upgraded pip was installed in this new specific location
!C:\Users\joey_\OneDrive\Documents\Home\Training\AI\Codecademy\Week 7\Assignment\rag_env\Scripts\python.exe -m pip install --upgrade pip

In [1]:
# Checking that the installed libraries now work - You'll notice that I get a warning. This is telling me that langchain-community is no longer being maintained because 
# it has all the tools bundled into . This is fine for the moment but if I wanted to created another AI agent in the future I would need to consider using the specific 
# tool langchain_openai import ChatOpenAI for example
try:
    import bs4
    import pandas
    import langchain
    import langchain_community
    import langchain_google_genai
    import faiss
    import streamlit
    import dotenv
    import sklearn
    print("All libraries are installed and are accessible in this notebook.")
except ImportError as e:
    print(f"Library missing: {e}")
    print("It is possible that the notebook is likely still connected to the 'base' environment instead of 'rag_env'.")

C:\Users\joey_\AppData\Local\Temp\ipykernel_25356\1160094315.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  import langchain_community


All libraries are installed and are accessible in this notebook.


## Part 2 - Creating the folder structures and implementing the data scrape

In [2]:
# Import the libraries for the data scrapeing
import os                      # Allows me to create folders using my operating system, which is Win10
import json                    # Allows me to read and write to and from JSON files. In this case I will be writing 'w' to a file at the end
import time                    # I don't want to overload the server so there will be 1 second gaps per webpage that is scraped
import urllib.request          # Using this allows me to grab data from the web, the colleges webpages and read them
from bs4 import BeautifulSoup  # Will enable the data to be cleaned of all of the HTML tags and then develop a binary tree for the words so they are searchable

In [3]:
# Create the necessary folder structure required by the assignment which is data/raw and data/processed
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

# The URLS are being held in a list. If I wanted to add more, I can at the end of the last website I listed
urls = [
    "https://burycollege.ac.uk/16-18",
    "https://burycollege.ac.uk/apprenticeships",
    "https://burycollege.ac.uk/adults",
    "https://burycollege.ac.uk/university-centre",
    "https://burycollege.ac.uk/employers",
    "https://burycollege.ac.uk/contact-us/contact-information",
    "https://burycollege.ac.uk/about-us/workforus",
    "https://burycollege.ac.uk/taking-teaching-further",
    "https://burycollege.ac.uk/greater-manchester-institute-of-technology",
    "https://burycollege.ac.uk/about-us/governance",
    "https://burycollege.ac.uk/about-us/our-vision",
    "https://burycollege.ac.uk/about-us/ofsted",
    "https://burycollege.ac.uk/about-us/term-dates",
    "https://burycollege.ac.uk/about-us/sustainability",
    "https://burycollege.ac.uk/support-for-schools",
    "https://burycollege.ac.uk/events",
    "https://burycollege.ac.uk/news"
]

# Creates a empty list. This will be populated with the scraped data, 1 at a time, using my for loop below
scraped_data = []

print("Starting scraper...")

# This loop takes in all the URLS and then one at a time places them into the variable URL to be processed before 
# moving on to the next URL
for url in urls:
    try:
        print(f"Scraping: {url}")
        
        # Using a standard User-Agent header to pretend it is a browser request
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as response:
            html = response.read()
            
        soup = BeautifulSoup(html, 'html.parser')
        
        # Strip away the html tags
        for element in soup(["script", "style", "nav", "footer", "header", "sidebar"]):
            element.decompose()
            
        # Extract clean, readable text
        clean_text = soup.get_text(separator=" ").strip()
        # Collapse multiple spaces or newlines into single spaces
        clean_text = " ".join(clean_text.split())
        
        # .append is adding each url and content for each page to the variable "scraped_data"
        scraped_data.append({
            "source_page": url,
            "content": clean_text
        })
        
        # Respect server rules by waiting 1 second between page hits. Helps to ensure we don;t get overloaded messages from the server
        time.sleep(1)
    
    #If there are errors, they will be stored in variable 'e' and then printed off
    except Exception as e:
        print(f"Failed to scrape {url}: {e}")

# Save the clean raw data to data/raw/scraped_pages.json
with open('data/raw/scraped_pages.json', 'w', encoding='utf-8') as f:
    json.dump(scraped_data, f, indent=4, ensure_ascii=False)

# Using the len function I can get a verification message telling me how many pages have been successfully scraped.
print(f"\nSuccess! Successfully scraped {len(scraped_data)} pages and saved to data/raw/scraped_pages.json")

Starting scraper...
Scraping: https://burycollege.ac.uk/16-18
Scraping: https://burycollege.ac.uk/apprenticeships
Scraping: https://burycollege.ac.uk/adults
Scraping: https://burycollege.ac.uk/university-centre
Scraping: https://burycollege.ac.uk/employers
Scraping: https://burycollege.ac.uk/contact-us/contact-information
Scraping: https://burycollege.ac.uk/about-us/workforus
Scraping: https://burycollege.ac.uk/taking-teaching-further
Scraping: https://burycollege.ac.uk/greater-manchester-institute-of-technology
Scraping: https://burycollege.ac.uk/about-us/governance
Scraping: https://burycollege.ac.uk/about-us/our-vision
Scraping: https://burycollege.ac.uk/about-us/ofsted
Scraping: https://burycollege.ac.uk/about-us/term-dates
Scraping: https://burycollege.ac.uk/about-us/sustainability
Scraping: https://burycollege.ac.uk/support-for-schools
Scraping: https://burycollege.ac.uk/events
Scraping: https://burycollege.ac.uk/news

Success! Successfully scraped 17 pages and saved to data/raw/

In [7]:
# Checking that the data scrape worked with the first three pages before I go on to create the Q&A document
import json # Allows me to read and write to and from JSON files. In this case, I'm just read ing from it using 'r'

# Load and inspect the scraped pages file
with open('data/raw/scraped_pages.json', 'r', encoding='utf-8') as f:
    debug_data = json.load(f)

# Again, using len and the debug_data variable from above, the system dynamically counts the number of pages (as .json files) that have been scraped)
print(f"Total pages found in JSON: {len(debug_data)}")

# Print out the character count and a snippet of the first 3 pages
for idx, page in enumerate(debug_data[:3]):
    url = page.get("source_page", "Unknown URL")
    content = page.get("content", "")
    print(f"\n--- Page {idx+1} ---")
    print(f"URL: {url}")
    print(f"Content Character Length: {len(content)}")
    print(f"Snippet: {content[:150]}...")

Total pages found in JSON: 17

--- Page 1 ---
URL: https://burycollege.ac.uk/16-18
Content Character Length: 3899
Snippet: School Leavers (16-18) - Bury College Skip to main content Skip to search Skip to navigation Open action menu Campus map Contact us Course guides Appl...

--- Page 2 ---
URL: https://burycollege.ac.uk/apprenticeships
Content Character Length: 6643
Snippet: Apprenticeships - Bury College Skip to main content Skip to search Skip to navigation Open action menu Campus map Contact us Course guides Apply now A...

--- Page 3 ---
URL: https://burycollege.ac.uk/adults
Content Character Length: 4884
Snippet: Adult courses - Bury College Skip to main content Skip to search Skip to navigation Open action menu Campus map Contact us Course guides Apply now Adu...


## Part 3 - Utilising the API Key

In [5]:
# Ensure Googles API key is found and works
from dotenv import load_dotenv # Enables me to secure my API key when called


In [7]:
# Run load_dotenv into the environment so my program can pick up my API key
load_dotenv()

# Retrieve the key to verify it loaded correctly
google_api_key = os.getenv("GOOGLE_API_KEY")

if google_api_key:
    print("API Key set successfully!")
else:
    print("Error with the API key")



API Key set successfully!


## Part 4 - Creating the Q+A document

In [11]:
# Load in the libraries needed for the Q&A generation
import json          # Allows me to read and write to and from JSON files. In this case I will be read 'r' from a file at the start
import pandas as pd  # Used to store the Q+A document as a dataframe to it can be saved as a .csv file at the end
import time          # Used to give the server time before the next page is checked
from langchain_google_genai import ChatGoogleGenerativeAI # Is the brain of the chatbot, manage behviour through the temperature
                                                          # and can understand the chat structure

In [13]:
# Link with Gemini and load in the JSON file with the data from the scraped pages to generate the Q&A

# Initialise Gemini using the correct active production model name
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0.3)

# Load scraped pages
with open('data/raw/scraped_pages.json', 'r', encoding='utf-8') as f:
    scraped_pages = json.load(f)

all_qa_pairs = []

print("Starting Synthetic Q/A Generation with production model...")

for idx, page in enumerate(scraped_pages):
    url = page["source_page"]
    content = page["content"]
    
    # Skip empty pages
    if len(content.strip()) < 200:
        continue
        
    print(f"[{idx+1}/{len(scraped_pages)}] Generating Q/As for: {url}")
    
    # Clean structured prompt for raw text JSON collection
    prompt = f"""You are an expert AI dataset generator. Read the text below from a college website and generate exactly 12 distinct, high-quality question and answer pairs.
The questions must look like real user queries from prospective students, parents, adult learners, employers or governors.

CRITICAL: Return ONLY a valid JSON object. Do not wrap it in ```json blocks. Do not write introductory or concluding text.

Follow this exact structure:
{{
  "qa_pairs": [
    {{"question": "Example question?", "answer": "Example answer based on the text."}}
  ]
}}

Context:
{content[:5000]}
"""
    
    max_retries = 3
    attempt = 0
    success = False
    
    while attempt < max_retries and not success:
        try:
            response = llm.invoke(prompt)
            
            # Handle cases where response.content is returned as a list of blocks instead of a string
            if isinstance(response.content, list):
                text_parts = []
                for part in response.content:
                    if isinstance(part, str):
                        text_parts.append(part)
                    elif isinstance(part, dict):
                        text_parts.append(part.get("text", ""))
                text_reply = "".join(text_parts).strip()
            else:
                text_reply = response.content.strip()
            
            # Clean any markdown block formatting if the model prints it anyway
            if text_reply.startswith("```"):
                text_reply = text_reply.split("```")[1]
                if text_reply.startswith("json"):
                    text_reply = text_reply[4:]
            
            data = json.loads(text_reply.strip())
            
            for pair in data.get("qa_pairs", []):
                all_qa_pairs.append({
                    "question": pair["question"],
                    "answer": pair["answer"],
                    "source_page": url
                })
                
            success = True  # Break the while loop
            time.sleep(4)   # Safe pacing between successful pages
            
        except Exception as e:
            error_msg = str(e)
            if "429" in error_msg or "RESOURCE_EXHAUSTED" in error_msg:
                attempt += 1
                print(f"   [Rate Limit Hit] Sliding window full. Backing off for 60s (Attempt {attempt}/{max_retries})...")
                time.sleep(60)  # Wait out the full sliding window
            else:
                print(f"   Skipped page due to non-rate-limit parsing error: {e}")
                break  # Break out of the while loop to move to the next page

# Convert data and save to CSV file
if all_qa_pairs:
    df_qa = pd.DataFrame(all_qa_pairs)
    df_qa.to_csv("data/processed/qa_dataset.csv", index=False, encoding='utf-8')
    print(f"\nSuccess! Generated {len(df_qa)} Q/A pairs and saved to data/processed/qa_dataset.csv")
else:
    print("\nGeneration failed. Let's make sure the API key is fully active.")

Starting Synthetic Q/A Generation with production model...
[1/17] Generating Q/As for: https://burycollege.ac.uk/16-18
[2/17] Generating Q/As for: https://burycollege.ac.uk/apprenticeships
[3/17] Generating Q/As for: https://burycollege.ac.uk/adults
[4/17] Generating Q/As for: https://burycollege.ac.uk/university-centre
[5/17] Generating Q/As for: https://burycollege.ac.uk/employers
[6/17] Generating Q/As for: https://burycollege.ac.uk/contact-us/contact-information
[7/17] Generating Q/As for: https://burycollege.ac.uk/about-us/workforus
[8/17] Generating Q/As for: https://burycollege.ac.uk/taking-teaching-further
[9/17] Generating Q/As for: https://burycollege.ac.uk/greater-manchester-institute-of-technology
[10/17] Generating Q/As for: https://burycollege.ac.uk/about-us/governance
[11/17] Generating Q/As for: https://burycollege.ac.uk/about-us/our-vision
[12/17] Generating Q/As for: https://burycollege.ac.uk/about-us/ofsted
[13/17] Generating Q/As for: https://burycollege.ac.uk/about

## Part 5 - Chunking the web scraped data, create a vector store for the Q+A and check distance scores

In [15]:
# Load in the libraires
import json # Allows me to read and write to and from JSON files. In this case I will be writing 'w' to a file at the end
import time # Added in so I can send the data in timed chunks as I'm using a free API from Google and my resources are limited.
import pandas as pd # Allows me to read my .csv file
from langchain_text_splitters import RecursiveCharacterTextSplitter # Allows the data scrapped to be chunked, stoping overloading
from langchain_core.documents import Document                       # Use metadata structures for FAISS to read
from langchain_google_genai import GoogleGenerativeAIEmbeddings     # Scraped data can be turned into vectors
from langchain_community.vectorstores import FAISS                  # Used instead of Chroma as my OneDrive was causing a conflict 
                                                                    # with it.

In [17]:
print("Connecting to Google Embeddings API...")
# Switch to the fresh daily quota bucket
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

# Load the newly aligned document store
persist_directory = "data/processed/faiss_db"
print("Loading full document vector database...")
doc_vector_store = FAISS.load_local(
    persist_directory, 
    embeddings, 
    allow_dangerous_deserialization=True
)

# Load the synthetic Q/A dataset 
qa_path = "data/processed/qa_dataset.csv" 
print(f"Loading synthetic Q/A dataset from: {qa_path}")
qa_df = pd.read_csv(qa_path)

qa_documents = []
for idx, row in qa_df.iterrows():
    doc = Document(
        page_content=str(row['question']),
        metadata={
            "answer": str(row['answer']),
            "source": str(row['source_page'])
        }
    )
    qa_documents.append(doc)

total_qa = len(qa_documents)
batch_size = 5
delay_seconds = 3

print(f"Building semantic search index for {total_qa} Q/A pairs safely in batches...")

print("-> Initializing Q/A batch 1...")
qa_vector_store = FAISS.from_documents(qa_documents[0:batch_size], embeddings)
time.sleep(delay_seconds)

for i in range(batch_size, total_qa, batch_size):
    batch = qa_documents[i:i + batch_size]
    
    current_batch_num = (i // batch_size) + 1
    total_batches = (total_qa + batch_size - 1) // batch_size
    print(f"-> Processing Q/A batch {current_batch_num}/{total_batches}...")
    
    max_retries = 5
    attempt = 0
    success = False
    
    while attempt < max_retries and not success:
        try:
            qa_vector_store.add_documents(batch)
            success = True
            if i + batch_size < total_qa:
                time.sleep(delay_seconds)
        except Exception as e:
            error_msg = str(e)
            if "429" in error_msg or "RESOURCE_EXHAUSTED" in error_msg:
                attempt += 1
                print(f"[Quota Hit] Rate limit reached. Pausing 60s (Attempt {attempt}/{max_retries})...")
                time.sleep(60)
            else:
                print(f"Critical Error: {e}")
                raise e

print("\n Both retrieval layers are loaded, aligned, and completely ready!")

Connecting to Google Embeddings API...
Loading full document vector database...
Loading synthetic Q/A dataset from: data/processed/qa_dataset.csv
Building semantic search index for 204 Q/A pairs safely in batches...
-> Initializing Q/A batch 1...
-> Processing Q/A batch 2/41...
-> Processing Q/A batch 3/41...
-> Processing Q/A batch 4/41...
-> Processing Q/A batch 5/41...
-> Processing Q/A batch 6/41...
-> Processing Q/A batch 7/41...
-> Processing Q/A batch 8/41...
-> Processing Q/A batch 9/41...
-> Processing Q/A batch 10/41...
-> Processing Q/A batch 11/41...
-> Processing Q/A batch 12/41...
-> Processing Q/A batch 13/41...
-> Processing Q/A batch 14/41...
-> Processing Q/A batch 15/41...
-> Processing Q/A batch 16/41...
-> Processing Q/A batch 17/41...
-> Processing Q/A batch 18/41...
-> Processing Q/A batch 19/41...
-> Processing Q/A batch 20/41...
-> Processing Q/A batch 21/41...
-> Processing Q/A batch 22/41...
-> Processing Q/A batch 23/41...
-> Processing Q/A batch 24/41...
->

## Part 6 - Load the model, upload the context to the chatbot and enable the formatting of the response

In [23]:
# Chatbot libraries
from langchain_google_genai import ChatGoogleGenerativeAI # Is the brain of the chatbot, manage behviour through the temperature
                                                          # and can understand the chat structure
from langchain_core.prompts import ChatPromptTemplate     # Allows me to program in the context behind the scenes
from langchain_core.output_parsers import StrOutputParser # Cleans the output so the user gets to see just the message
                                                          # rather than the the code and meta data which is attached to the output.

In [25]:
# Initialise the Gemini LLM (Updated to current active model)
print("Initializing Gemini LLM...")
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash", 
    temperature=0.2 # Keeps the model factual and grounded in your retrieved text
)

# Design the System Prompt layout required by the assignment brief
prompt_template = ChatPromptTemplate.from_messages([
    ("system", (
        "You are an intelligent, helpful, and official AI assistant for Bury College.\n"
        "Your task is to answer the user's question accurately using ONLY the provided context below.\n\n"
        "=== CONTEXT ===\n"
        "{context}\n"
        "===============\n\n"
        "Guidelines:\n"
        "- Base your answer strictly on the provided context. If the answer cannot be found in the context, "
        "politely state that you do not have that information and suggest contacting the college directly.\n"
        "- Maintain a professional, welcoming tone.\n"
        "- Do not mention or reveal the underlying data sources, brackets, or database names in your conversational text."
    )),
    ("human", "{question}")
])

# Create the conversational chain
chain = prompt_template | llm | StrOutputParser()

# Define the Complete Chatbot Engine
def bury_college_chatbot(user_question):
    # Step A: Retrieve relevant context using our hybrid logic
    retrieval_data = hybrid_retrieve(user_question)
    
    # Step B: Pass the text context and the question into our LLM chain
    print("🤖 Gemini is generating your answer based on retrieved facts...")
    llm_response = chain.invoke({
        "context": retrieval_data["context"],
        "question": user_question
    })
    
    # Step C: Format and print the final output with proper assignment citations
    print("\n" + "="*60)
    print("🎓 BURY COLLEGE ASSISTANT RESPONSE:")
    print("="*60)
    print(llm_response)
    print("="*60)
    print(f"Source Type: {retrieval_data['source_type']}")
    print(f"Verified Source URL(s): {retrieval_data['source']}")
    print("="*60 + "\n")

Initializing Gemini LLM...


## Part 7 - Ensuring the FAQ database is saved on my hard drive

In [27]:
qa_vector_store.save_local("data/processed/faiss_qa_db")
print("QA Vector Store saved successfully!")

QA Vector Store saved successfully!


## Part 8 - Prompt to check that questions can be answered as expected

In [29]:
bury_college_chatbot("What courses are available?")


🔎 [User Query]: 'What courses are available?'
-> [Stage 1 - Q/A Match] Nearest question found with distance score: 0.6146
🔄 Confidence low or no exact match. Falling back to full document chunk search...
🤖 Gemini is generating your answer based on retrieved facts...

🎓 BURY COLLEGE ASSISTANT RESPONSE:
Bury College offers a wide range of courses across various subject areas. Based on our current listings, here are the courses available:

**Health and Social Care**
* Health and Social Care (Level 2 and Level 3)

**Catering and Hospitality**
* Professional Cookery (Level 2 and Level 3)
* Hospitality and Catering (Level 2)

**Construction and the Built Environment**
* Bricklaying (Level 2)
* Carpentry and Joinery (Level 2 and Level 3)
* Electrical Installation (Level 2 and Level 3)
* Plastering (Level 2)
* Plumbing (Level 2 and Level 3)

**English and Maths**
* GCSE English (Level 2)
* Functional Skills English (Level 2)
* GCSE Maths (Level 2)
* Functional Skills Maths (Level 2)

**Specia

In [50]:
bury_college_chatbot("Can I earn money while studying an apprenticeship?")


🔎 [User Query]: 'Can I earn money while studying an apprenticeship?'
-> [Stage 1 - Q/A Match] Nearest question found with distance score: 0.3493
🎯 High-confidence match found in Q/A dataset! Bypassing document search.
🤖 Gemini is generating your answer based on retrieved facts...

🎓 BURY COLLEGE ASSISTANT RESPONSE:
Yes, you can! An apprenticeship at Bury College is a job where you will have the opportunity to learn new skills, gain experience, and get paid while working towards your chosen career.
Source Type: Curated Q/A Pair
Verified Source URL(s): https://burycollege.ac.uk/16-18



In [44]:
bury_college_chatbot("What are the different study pathways available for 16-18 year old school leavers at Bury college?")


🔎 [User Query]: 'What are the different study pathways available for 16-18 year old school leavers at Bury college?'
-> [Stage 1 - Q/A Match] Nearest question found with distance score: 0.2561
🎯 High-confidence match found in Q/A dataset! Bypassing document search.
🤖 Gemini is generating your answer based on retrieved facts...

🎓 BURY COLLEGE ASSISTANT RESPONSE:
Welcome to Bury College! 

For 16-18 year old school leavers, we offer four distinct study pathways:

* **A Levels**
* **Vocational Programmes**
* **T Levels**
* **Apprenticeships**
Source Type: Curated Q/A Pair
Verified Source URL(s): https://burycollege.ac.uk/16-18



## Part 9 - Streamlit
The line of code below installs Steamlit to my rag_env enviroment. A seperate Python file called app.py is present in this folder

In [ ]:
!rag_env\Scripts\pip install streamlit

### Generate the .txt file

In [3]:
pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.
